# 07 · Predictive Analysis: Classification

Reproduce stratified train/test evaluation and compare five classifiers with 5-fold stratified cross-validation. Preprocessing is included in the Pipeline to avoid fitting encoders/scalers on holdout data.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
ROOT=Path.cwd().parent if Path.cwd().name=="notebooks" else Path.cwd()
df=pd.read_csv(ROOT/"data"/"falcon9_labeled.csv")
num=["PayloadMass","Flights","Block","ReusedCount"]; cat=["Orbit","LaunchSite","GridFins","Reused","Legs"]
X=df[num+cat]; y=df["Class"]
pre=ColumnTransformer([("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),cat)])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)

In [ ]:
models={"Logistic Regression":(LogisticRegression(max_iter=3000),{"model__C":[.1,1,10]}),"SVM":(SVC(probability=True,random_state=42),{"model__C":[.1,1,10],"model__kernel":["linear","rbf"]}),"Decision Tree":(DecisionTreeClassifier(random_state=42),{"model__max_depth":[2,4,6,None],"model__min_samples_leaf":[1,2,4]}),"KNN":(KNeighborsClassifier(),{"model__n_neighbors":[3,5,7],"model__weights":["uniform","distance"]}),"Random Forest":(RandomForestClassifier(random_state=42),{"model__n_estimators":[200],"model__max_depth":[3,5,None],"model__min_samples_leaf":[1,2]})}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42); results=[]
for name,(est,grid) in models.items():
    gs=GridSearchCV(Pipeline([("prep",pre),("model",est)]),grid,cv=cv,scoring="accuracy",n_jobs=-1).fit(Xtr,ytr)
    p=gs.predict(Xte); prob=gs.predict_proba(Xte)[:,1]
    results.append({"model":name,"cv_accuracy":gs.best_score_,"test_accuracy":accuracy_score(yte,p),"precision":precision_score(yte,p),"recall":recall_score(yte,p),"f1":f1_score(yte,p),"roc_auc":roc_auc_score(yte,prob),"best_params":gs.best_params_})
pd.DataFrame(results).sort_values("test_accuracy",ascending=False)

Interpret results cautiously: the held-out test set contains only 18 launches; one record changes a metric materially. Class 0 includes no-attempt and other coded outcomes.